In [ ]:
import os
import glob
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
import pynapple as nap
from spatial_manifolds.toroidal import *
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.data.binning import get_bin_config
from spatial_manifolds.mlencoding import *
from spatial_manifolds.circular_decoder import circular_decoder, cross_validate_decoder, cross_validate_decoder_time, circular_nanmean
from spatial_manifolds.data.curation import curate_clusters
from scipy.stats import zscore
from spatial_manifolds.util import gaussian_filter_nan
from spatial_manifolds.predictive_grid import compute_travel_projected, wrap_list
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

fig_path = '/Users/harryclark/Documents/figs/FIGURE1/'
source_path = '/Users/harryclark/Downloads/COHORT12/'
locations = pd.read_csv('/Users/harryclark/Downloads/all_cluster_brain_locations_chris.csv')
locations['coord_SCs_x'] = locations['coord_SCs_x'] * -1

cell_classifications = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv')

## XGBoost shank assay — GC predicted by NGS cells per shank

### What this analysis does

For every session containing at least one grid cell and at least one non-grid spatial (NGS) cell, we fit XGBoost models to predict each **target grid cell's** firing rate in VR. Two covariate conditions are tested:

| `covariate_type` | Covariates |
|---|---|
| `pos` | Position alone (baseline, fit once per GC) |
| `pos+ngs_shank` | Position + all NGS cells on a given shank (fit once per GC × shank) |

Shank IDs (0–3) are assigned from `probe_x` (electrode x-position on the probe) via `reconstruct_shank_id`. Only shanks that contain at least one NGS cell contribute a `pos+ngs_shank` row for a given GC.

This lets us ask whether NGS activity on a particular shank — including the same shank as the GC — predicts the GC's VR firing above and beyond position alone, enabling shank-by-shank population comparison.

---

### Resulting dataframe — `results_df`

Each row is one XGBoost model fit. Columns:

| Column | Description |
|---|---|
| `mouse` | Mouse ID |
| `day` | Recording day |
| `target_cluster_id` | Cluster ID of the GC being predicted |
| `target_shank_id` | Shank ID of the target GC (0–3) |
| `cov_shank_id` | Shank ID of the NGS covariate pool (`NaN` for baseline) |
| `n_ngs_cells` | Number of NGS cells used as covariates (`NaN` for baseline) |
| `covariate_type` | `pos` or `pos+ngs_shank` |
| `pR2_cv` | Cross-validated Poisson pseudo-R² (mean across CV folds) |
| `mean_dist_to_ngs_um` | Mean 3-D Euclidean distance (µm) from GC to NGS cells on that shank (`NaN` for baseline) |


In [ ]:

# XGBoost model parameters
nfilters = 5
history_length = 1000  # ms

xgb_history = MLencoding(
    tunemodel='xgboost',
    cov_history=True,
    spike_history=False,
    window=time_bs,
    n_filters=nfilters,
    max_time=history_length,
)

all_rows = []

cache_dir = '/Users/harryclark/Documents/spatial-manifolds/data/gc_ngs_shank_xgboost_cache'
os.makedirs(cache_dir, exist_ok=True)

# Sessions with at least 1 GC and at least 1 NGS cell
gc_sessions  = cell_classifications[cell_classifications['cell_type'] == 'GC'][['mouse', 'day']].drop_duplicates()
ngs_sessions = cell_classifications[cell_classifications['cell_type'] == 'NG'][['mouse', 'day']].drop_duplicates()
valid_sessions = gc_sessions.merge(ngs_sessions, on=['mouse', 'day'])
print(f"{len(valid_sessions)} sessions have at least 1 GC and 1 NGS cell.")

expected_shanks = {0, 1, 2, 3}

for _, sess_row in valid_sessions.iterrows():
    mouse_id = int(sess_row['mouse'])
    day_id   = int(sess_row['day'])
    print(f"\n=== M{mouse_id} D{day_id:02d} ===")

    # ── Session-level cache check: skip loading if all GCs are already cached ──
    _sess_gcs_check = cell_classifications[
        (cell_classifications['mouse'] == mouse_id) &
        (cell_classifications['day']   == day_id) &
        (cell_classifications['cell_type'] == 'GC')
    ]
    _gc_cache_files = {
        int(cid): os.path.join(cache_dir, f'M{mouse_id}_D{day_id:02d}_C{int(cid)}.csv')
        for cid in _sess_gcs_check['cluster_id']
    }
    if len(_gc_cache_files) > 0 and all(os.path.exists(f) for f in _gc_cache_files.values()):
        for cid, cache_file in _gc_cache_files.items():
            all_rows.extend(pd.read_csv(cache_file).to_dict('records'))
        print("  All GCs cached — skipping session load.")
        continue

    # Load VR spike trains and behaviour
    tcs, tcs_time, _, last_ephys_bin, beh, clusters = compute_vr_tcs(
        mouse_id, day_id,
        apply_zscore=False, apply_guassian_filter=False,
        source_path=source_path,
    )

    last_ephys_time_bin = clusters[clusters.index[0]].count(
        bin_size=time_bs, time_units='ms'
    ).index[-1]
    ep = nap.IntervalSet(start=0, end=last_ephys_time_bin, time_units='s')

    dt_in_time  = np.array(
        beh['travel'].bin_average(bin_size=time_bs, time_units='ms', ep=ep)
        - ((beh['trial_number'][0] - 1) * tl)
    )
    pos_in_time = dt_in_time % tl
    if np.any(np.isnan(pos_in_time)):
        pos_in_time = pd.Series(dt_in_time).ffill().bfill().values % tl

    # ── Session cell tables ────────────────────────────────────────────────────
    sess_gcs = cell_classifications[
        (cell_classifications['mouse'] == mouse_id) &
        (cell_classifications['day']   == day_id) &
        (cell_classifications['cell_type'] == 'GC')
    ].copy()
    sess_ngs = cell_classifications[
        (cell_classifications['mouse'] == mouse_id) &
        (cell_classifications['day']   == day_id) &
        (cell_classifications['cell_type'] == 'NG')
    ].copy()

    sess_gcs['cluster_id'] = sess_gcs['cluster_id'].astype(int)
    sess_ngs['cluster_id'] = sess_ngs['cluster_id'].astype(int)

    # ── Assign shank IDs from probe_x ─────────────────────────────────────────
    sess_gcs = reconstruct_shank_id(sess_gcs, mouse_id, colname='probe_x')
    sess_ngs = reconstruct_shank_id(sess_ngs, mouse_id, colname='probe_x')

    # Group NGS cells by shank — keep only shanks present in tcs_time
    ngs_by_shank = {
        shank: grp['cluster_id'].values.astype(int)
        for shank, grp in sess_ngs.groupby('shank_id')
        if any(c in tcs_time for c in grp['cluster_id'].astype(int))
    }

    # Require NGS cells on every shank
    if set(ngs_by_shank.keys()) != expected_shanks:
        print(f"  NGS cells not on all shanks (present: {sorted(ngs_by_shank.keys())}) — skipping.")
        continue

    print(f"  GCs: {len(sess_gcs)}  |  NGS shanks: { {s: len(v) for s,v in ngs_by_shank.items()} }")

    # ── Loop over each target GC ───────────────────────────────────────────────
    for _, gc_row in sess_gcs.iterrows():
        target_id   = int(gc_row['cluster_id'])
        target_shank = int(gc_row['shank_id'])

        # ── GC-level cache check ───────────────────────────────────────────────
        cache_file = os.path.join(cache_dir, f'M{mouse_id}_D{day_id:02d}_C{target_id}.csv')
        if os.path.exists(cache_file):
            print(f"    GC {target_id} (shank {target_shank}): loading from cache.")
            all_rows.extend(pd.read_csv(cache_file).to_dict('records'))
            continue

        if target_id not in tcs_time:
            continue

        y = np.array(tcs_time[target_id])
        T = len(y)
        pos = pos_in_time[:T]
        if len(pos) < T:
            pos = np.pad(pos, (0, T - len(pos)), mode='edge')

        tgt_x = float(gc_row['SC_x'])
        tgt_y = float(gc_row['SC_y'])
        tgt_z = float(gc_row['SC_z'])

        gc_rows = []

        # ── Baseline: position only ────────────────────────────────────────────
        _, pR2_cv = xgb_history.fit_cv(pos[:, None], y, verbose=0, continuous_folds=True)
        gc_rows.append(dict(
            mouse=mouse_id, day=day_id,
            target_cluster_id=target_id,
            target_shank_id=target_shank,
            cov_shank_id=np.nan,
            n_ngs_cells=np.nan,
            covariate_type='pos',
            pR2_cv=float(np.nanmean(pR2_cv)),
            mean_dist_to_ngs_um=np.nan,
        ))

        # ── One model per NGS shank ────────────────────────────────────────────
        for cov_shank, ngs_cids in ngs_by_shank.items():

            # Keep only NGS cells present in tcs_time
            valid_ngs = [c for c in ngs_cids if c in tcs_time]
            if not valid_ngs:
                continue

            # Stack NGS spike trains
            ngs_mat = np.vstack([
                np.pad(
                    np.array(tcs_time[c])[:T],
                    (0, max(0, T - len(np.array(tcs_time[c])[:T]))),
                    mode='constant'
                )
                for c in valid_ngs
            ]).T  # shape (T, n_ngs)

            x = np.column_stack([pos, ngs_mat])

            # Mean Euclidean distance from GC to each NGS cell on this shank
            ngs_coords = sess_ngs[sess_ngs['cluster_id'].isin(valid_ngs)][['SC_x', 'SC_y', 'SC_z']]
            dists = np.sqrt(
                (ngs_coords['SC_x'].values.astype(float) - tgt_x)**2 +
                (ngs_coords['SC_y'].values.astype(float) - tgt_y)**2 +
                (ngs_coords['SC_z'].values.astype(float) - tgt_z)**2
            )
            mean_dist = float(np.nanmean(dists))

            _, pR2_cv = xgb_history.fit_cv(x, y, verbose=0, continuous_folds=True)
            gc_rows.append(dict(
                mouse=mouse_id, day=day_id,
                target_cluster_id=target_id,
                target_shank_id=target_shank,
                cov_shank_id=int(cov_shank),
                n_ngs_cells=len(valid_ngs),
                covariate_type='pos+ngs_shank',
                pR2_cv=float(np.nanmean(pR2_cv)),
                mean_dist_to_ngs_um=mean_dist,
            ))

        # ── Save this GC's rows to cache ──────────────────────────────────────
        pd.DataFrame(gc_rows).to_csv(cache_file, index=False)
        all_rows.extend(gc_rows)
        print(f"    GC {target_id} (shank {target_shank}): done.")

results_df = pd.DataFrame(all_rows)
print(f"\nTotal rows collected: {len(results_df)}")
print(f"covariate_type counts:\n{results_df['covariate_type'].value_counts()}")
results_df.head(10)


66 sessions have at least 1 GC and 1 NGS cell.

=== M20 D14 ===
  GCs: 4  |  NGS shanks: {0: 2, 1: 32, 2: 54, 3: 21}


In [ ]:
save_path = '/Users/harryclark/Documents/spatial-manifolds/data/gc_ngs_shank_xgboost.csv'
results_df.to_csv(save_path, index=False)
print(f"Saved {len(results_df)} rows → {save_path}")


In [ ]:

import statsmodels.formula.api as smf

# ── Prepare identifiers for the nested hierarchy ───────────────────────────
lmm_df = results_df.copy()
lmm_df['mouse_day'] = (
    lmm_df['mouse'].astype(str) + '_' + lmm_df['day'].astype(str)
)
lmm_df['mouse_day_cluster'] = (
    lmm_df['mouse_day'] + '_' + lmm_df['target_cluster_id'].astype(str)
)

# Treat cov_shank_id as categorical so shank labels are compared versus a reference
lmm_df['cov_shank_id'] = lmm_df['cov_shank_id'].astype('Int64')

# ── Fit a LMM for a given covariate_type subset ───────────────────────────
def fit_lmm(df_sub, formula, groups_col='mouse', label=''):
    """Fit a Mixed Linear Model and return the result object.

    Hierarchy:
      Level 1 – mouse (groups), intercept-only random effect (re_formula='1')
      Level 2 – session (mouse_day) via vc_formula
      Level 3 – cell    (mouse_day_cluster) via vc_formula
    """
    md = smf.mixedlm(
        formula,
        data=df_sub,
        groups=df_sub[groups_col],
        re_formula="1",
        vc_formula={
            "mouse_day":         "0 + C(mouse_day)",
            "mouse_day_cluster": "0 + C(mouse_day_cluster)",
        },
    )
    result = md.fit(method='lbfgs', maxiter=1000)
    print(f"\n{'='*60}")
    print(f"  LMM: {label}")
    print(f"  Formula : {formula}")
    print(f"  N obs   : {len(df_sub)}")
    print(f"{'='*60}")
    print(result.summary())
    return result


lmm_results = {}

# ── pos+ngs_shank: fixed effects = shank identity + pool size ─────────────
df_cov = lmm_df[lmm_df['covariate_type'] == 'pos+ngs_shank'].dropna(
    subset=['cov_shank_id', 'n_ngs_cells', 'pR2_cv']
).copy()

lmm_results['pos+ngs_shank'] = fit_lmm(
    df_cov,
    formula='pR2_cv ~ C(cov_shank_id) + n_ngs_cells',
    label='pos+ngs_shank',
)

# ── pos (baseline): intercept-only — characterises GC-level variability ───
df_pos = lmm_df[lmm_df['covariate_type'] == 'pos'].dropna(
    subset=['pR2_cv']
).copy()

lmm_results['pos'] = fit_lmm(
    df_pos,
    formula='pR2_cv ~ 1',
    label='pos (baseline, intercept-only)',
)

# ── Save fixed-effects tables ──────────────────────────────────────────────
fe_rows = []
for cov_type, res in lmm_results.items():
    fe = res.summary().tables[1]
    fe_df = pd.DataFrame(fe.data[1:], columns=fe.data[0])
    fe_df.insert(0, 'covariate_type', cov_type)
    fe_rows.append(fe_df)

fe_table = pd.concat(fe_rows, ignore_index=True)
lmm_save_path = '/Users/harryclark/Documents/spatial-manifolds/data/gc_ngs_shank_lmm_fixed_effects.csv'
fe_table.to_csv(lmm_save_path, index=False)
print(f"\nFixed-effects table saved → {lmm_save_path}")
fe_table


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

# ── Compute per-cell Δ pR² (pos+ngs_shank minus pos baseline) ─────────────
# Baseline pR² is unique per (mouse, day, target_cluster_id)
baseline = (
    results_df[results_df['covariate_type'] == 'pos']
    [['mouse', 'day', 'target_cluster_id', 'pR2_cv']]
    .rename(columns={'pR2_cv': 'pR2_baseline'})
)
cov_df = results_df[results_df['covariate_type'] == 'pos+ngs_shank'].copy()
cov_df = cov_df.merge(baseline, on=['mouse', 'day', 'target_cluster_id'], how='left')
cov_df['delta_pR2'] = cov_df['pR2_cv'] - cov_df['pR2_baseline']

# Label whether the NGS shank is the same as the GC's shank
cov_df['same_shank'] = cov_df['cov_shank_id'] == cov_df['target_shank_id']
cov_df['shank_label'] = cov_df['same_shank'].map({True: 'Same shank', False: 'Different shank'})

shank_palette = {0: '#4c72b0', 1: '#dd8452', 2: '#55a868', 3: '#c44e52'}


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 1 — Violin: raw pR² by covariate shank, with per-cell baseline overlaid
#
# Shows the overall prediction quality for each shank's NGS pool.
# The dashed line marks the mean pos baseline so you can see whether any
# shank improves on the baseline at all.
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(5, 4))
sns.violinplot(
    data=cov_df, x='cov_shank_id', y='pR2_cv',
    palette=shank_palette, cut=0, inner='box', ax=ax
)
mean_baseline = baseline['pR2_baseline'].mean()
ax.axhline(mean_baseline, color='k', linestyle='--', linewidth=1.2, label='Mean pos baseline')
ax.set_xlabel('NGS covariate shank')
ax.set_ylabel('pR² (cross-validated)')
ax.set_title('Plot 1 — Absolute pR² by NGS shank')
ax.legend(fontsize=8)
fig.tight_layout()
plt.savefig(fig_path + 'ngs_shank_abs_pR2_violin.pdf', dpi=300)
plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 2 — Violin: Δ pR² by covariate shank
#
# Subtracts each cell's own position-only baseline, isolating the unique
# contribution of that shank's NGS pool.  Values > 0 indicate genuine
# improvement beyond position coding.
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(5, 4))
sns.violinplot(
    data=cov_df, x='cov_shank_id', y='delta_pR2',
    palette=shank_palette, cut=0, inner='box', ax=ax
)
ax.axhline(0, color='k', linestyle='--', linewidth=1.2)
ax.set_xlabel('NGS covariate shank')
ax.set_ylabel('Δ pR² (pos+NGS − pos)')
ax.set_title('Plot 2 — Δ pR² by NGS shank (baseline-corrected)')
fig.tight_layout()
plt.savefig(fig_path + 'ngs_shank_delta_pR2_violin.pdf', dpi=300)
plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 3 — Strip + box: same shank vs all different shanks
#
# Collapses across shank IDs to ask the binary question:
# does the NGS pool from the GC's own shank predict better than
# NGS pools from the other shanks?
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(3.5, 4))
sns.boxplot(
    data=cov_df, x='shank_label', y='delta_pR2',
    order=['Same shank', 'Different shank'],
    palette={'Same shank': '#e64b35', 'Different shank': '#4dbbd5'},
    width=0.4, fliersize=0, ax=ax
)
sns.stripplot(
    data=cov_df, x='shank_label', y='delta_pR2',
    order=['Same shank', 'Different shank'],
    palette={'Same shank': '#e64b35', 'Different shank': '#4dbbd5'},
    alpha=0.3, jitter=True, size=3, ax=ax
)
ax.axhline(0, color='k', linestyle='--', linewidth=1.2)
ax.set_xlabel('')
ax.set_ylabel('Δ pR² (pos+NGS − pos)')
ax.set_title('Plot 3 — Same vs different shank')
fig.tight_layout()
plt.savefig(fig_path + 'ngs_same_vs_diff_shank.pdf', dpi=300)
plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 4 — Heatmap: mean Δ pR² for each target_shank × cov_shank combination
#
# Shows the spatial pattern of NGS-GC interactions across the probe.
# Strong diagonal values would indicate that cells on the same shank
# co-predict each other; off-diagonal values show cross-shank influence.
# ═══════════════════════════════════════════════════════════════════════════════
pivot = cov_df.pivot_table(
    values='delta_pR2',
    index='target_shank_id',
    columns='cov_shank_id',
    aggfunc='mean'
)
fig, ax = plt.subplots(figsize=(4.5, 3.8))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r',
               vmin=-np.nanmax(np.abs(pivot.values)),
               vmax= np.nanmax(np.abs(pivot.values)))
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns.astype(int))
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index.astype(int))
ax.set_xlabel('NGS covariate shank')
ax.set_ylabel('Target GC shank')
ax.set_title('Plot 4 — Mean Δ pR² (target × covariate shank)')
plt.colorbar(im, ax=ax, label='Mean Δ pR²')

# Annotate each cell with its value
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8)

fig.tight_layout()
plt.savefig(fig_path + 'ngs_shank_delta_pR2_heatmap.pdf', dpi=300)
plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
# PLOT 5 — Scatter: Δ pR² vs mean anatomical distance to NGS pool
#
# Tests whether spatial proximity between the GC and the NGS pool
# predicts how much those NGS cells improve GC decoding.
# Points are coloured by covariate shank ID.
# ═══════════════════════════════════════════════════════════════════════════════
plot5_df = cov_df.dropna(subset=['mean_dist_to_ngs_um', 'delta_pR2'])
fig, ax = plt.subplots(figsize=(5, 4))
for shank_id, grp in plot5_df.groupby('cov_shank_id'):
    ax.scatter(
        grp['mean_dist_to_ngs_um'] / 1000,  # convert µm → mm for readability
        grp['delta_pR2'],
        s=15, alpha=0.4, label=f'Shank {int(shank_id)}',
        color=shank_palette.get(int(shank_id), 'grey')
    )

# Overall linear trend
from numpy.polynomial.polynomial import polyfit as polyfit_
x_all = plot5_df['mean_dist_to_ngs_um'].values / 1000
y_all = plot5_df['delta_pR2'].values
mask = np.isfinite(x_all) & np.isfinite(y_all)
if mask.sum() > 2:
    coefs = np.polyfit(x_all[mask], y_all[mask], 1)
    x_line = np.linspace(x_all[mask].min(), x_all[mask].max(), 200)
    ax.plot(x_line, np.polyval(coefs, x_line), 'k-', linewidth=1.5,
            label=f'Trend (slope={coefs[0]:.4f})')

ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_xlabel('Mean distance to NGS pool (mm)')
ax.set_ylabel('Δ pR² (pos+NGS − pos)')
ax.set_title('Plot 5 — Δ pR² vs anatomical distance to NGS pool')
ax.legend(fontsize=8, markerscale=1.5)
fig.tight_layout()
plt.savefig(fig_path + 'ngs_shank_delta_pR2_vs_dist.pdf', dpi=300)
plt.show()

print("All plots saved.")
